# End to End Supply Chain Analysis 

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from warnings import filterwarnings
filterwarnings("ignore")

In [3]:
data = pd.read_csv("SupplyChain.csv", encoding="latin-1")
# print(data.head())
data

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.750000,0,1/15/2018 11:24,Standard Class
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180514,CASH,4,4,40.000000,399.980011,Shipping on time,0,45,Fishing,Brooklyn,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 3:40,Standard Class
180515,DEBIT,3,2,-613.770019,395.980011,Late delivery,1,45,Fishing,Bakersfield,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/19/2016 1:34,Second Class
180516,TRANSFER,5,4,141.110001,391.980011,Late delivery,1,45,Fishing,Bristol,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/20/2016 21:00,Standard Class
180517,PAYMENT,3,4,186.229996,387.980011,Advance shipping,0,45,Fishing,Caguas,...,NaN,1004,45,NaN,http://images.acmesports.sports/Field+%26+Stre...,Field & Stream Sportsman 16 Gun Fire Safe,399.980011,0,1/18/2016 20:18,Standard Class


## Exploratory Data Analysis(EDA)

In [4]:
# Checking number of rows and columns
print("rows, columns:", data.shape)

# Number of columns
print("\nColumns:")
print(data.columns)

# Counting number of duplicated values.
print("\nNum duplicates:", data.duplicated().sum())

# Missing values from top 20 data.Late_delivery_risk
print("\nMissing values(top 20):")
dup = data.isnull().sum().sort_values(ascending=False).head(20)
dup

rows, columns: (180519, 53)

Columns:
Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Benefit per order', 'Sales per customer', 'Delivery Status',
       'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City',
       'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id',
       'Customer Lname', 'Customer Password', 'Customer Segment',
       'Customer State', 'Customer Street', 'Customer Zipcode',
       'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market',
       'Order City', 'Order Country', 'Order Customer Id',
       'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price', 'Order Item Profit Ratio',
       'Order Item Quantity', 'Sales', 'Order Item Total',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Order Zipcode', 'Product Card Id', 'Pr

Product Description         180519
Order Zipcode               155679
Customer Lname                   8
Customer Zipcode                 3
Type                             0
Order Profit Per Order           0
Order Item Cardprod Id           0
Order Item Discount              0
Order Item Discount Rate         0
Order Item Id                    0
Order Item Product Price         0
Order Item Profit Ratio          0
Order Item Quantity              0
Sales                            0
Order Item Total                 0
Order Region                     0
order date (DateOrders)          0
Order State                      0
Order Status                     0
Product Card Id                  0
dtype: int64

## Data cleaning

In [5]:
columns_to_drop = ['Latitude', 'Longitude', 'Market','Product Description',
       'Order City', 'Order Country', 'Order Customer Id','Order Id', 'Order Item Cardprod Id',
       'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id',
       'Order Item Product Price','Order Item Quantity', 'Sales', 'Order Item Total','Benefit per order','Benefit per order','Product Image',"Order Zipcode",'Customer Lname','Customer Zipcode','Customer Email','Customer Password']

# droppig columns that are either fully missing, redundant, or have only one value(and thus no variance)
data = data.drop(columns=columns_to_drop)

# Keeps only the TRUE value, ie the one with shipping_canceled is removed
data = data[data['Delivery Status'] != "Shipping_canceled"]
data


# Loop through both date columns.
# Convert them from text to datetime format.
# Invalid dates become NaT instead of causing errors.
# Dates are interpreted as Month/Day/Year format.
# Standard date conversion.
for c in ["order date (DateOrders)", 'shipping date (DateOrders)']:
    data[c] = pd.to_datetime(
        data[c],
        errors="coerce",
        dayfirst=False
        )

# after data cleaning, let's check the overview again to see how the dataset has changed.

print("Rows, Columns:", data.shape)
print("\nMissing values(top 5):")
print(data.isnull().sum().sort_values(ascending=False).head(5))
# If we sort the data in descending order then the column with more null value will be at top.

Rows, Columns: (180519, 30)

Missing values(top 5):
Type                          0
Days for shipping (real)      0
shipping date (DateOrders)    0
Product Status                0
Product Price                 0
dtype: int64


In [6]:
data.columns

Index(['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)',
       'Sales per customer', 'Delivery Status', 'Late_delivery_risk',
       'Category Id', 'Category Name', 'Customer City', 'Customer Country',
       'Customer Fname', 'Customer Id', 'Customer Segment', 'Customer State',
       'Customer Street', 'Department Id', 'Department Name',
       'order date (DateOrders)', 'Order Item Profit Ratio',
       'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status',
       'Product Card Id', 'Product Category Id', 'Product Name',
       'Product Price', 'Product Status', 'shipping date (DateOrders)',
       'Shipping Mode'],
      dtype='object')

In [7]:
# Value counts for categorical columns with low cardinality.
for col in data.columns:
    if data[col].nunique() < 10:    # Number of unique value less than 10.
        print(f"\n{col} value counts:")
        print(data[col].value_counts())


Type value counts:
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

Days for shipping (real) value counts:
Days for shipping (real)
2    56618
3    28765
6    28723
4    28513
5    28163
0     5080
1     4657
Name: count, dtype: int64

Days for shipment (scheduled) value counts:
Days for shipment (scheduled)
4    107752
2     35216
1     27814
0      9737
Name: count, dtype: int64

Delivery Status value counts:
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Late_delivery_risk value counts:
Late_delivery_risk
1    98977
0    81542
Name: count, dtype: int64

Customer Country value counts:
Customer Country
EE. UU.        111146
Puerto Rico     69373
Name: count, dtype: int64

Customer Segment value counts:
Customer Segment
Consumer       93504
Corporate      54789
Home Office    32226
Name: count, dtype: int64

Order Status value c

In [19]:
# Calculating order processing time and delay.
data["Order Processing Time"] = (data['shipping date (DateOrders)'] - data["order date (DateOrders)"]).dt.days
data["Delay"] = data['Order Processing Time'] - data["Days for shipment (scheduled)"]
data['Is_Delayed'] = data['Delay'] > 0
data["order_day"] = data["order date (DateOrders)"].dt.day_name()
data["order_month"] = data["order date (DateOrders)"].dt.month
data["order_hour"] = data["order date (DateOrders)"].dt.hour
data.describe

<bound method NDFrame.describe of             Type  Days for shipping (real)  Days for shipment (scheduled)  \
0          DEBIT                         3                              4   
1       TRANSFER                         5                              4   
2           CASH                         4                              4   
3          DEBIT                         3                              4   
4        PAYMENT                         2                              4   
...          ...                       ...                            ...   
180514      CASH                         4                              4   
180515     DEBIT                         3                              2   
180516  TRANSFER                         5                              4   
180517   PAYMENT                         3                              4   
180518   PAYMENT                         4                              4   

        Sales per customer   Delivery Sta

In [13]:
data.head(30)

,Type,Days for shipping (real),Days for shipment (scheduled),Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,Customer Country,...,Product Card Id,Product Category Id,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode,Order Processing Time,Delay,Is_Delayed
0,DEBIT,3,4,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,...,1360,73,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class,3,-1,False
1,TRANSFER,5,4,311.359985,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,...,1360,73,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class,5,1,True
2,CASH,4,4,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,EE. UU.,...,1360,73,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class,4,0,False
3,DEBIT,3,4,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,EE. UU.,...,1360,73,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class,3,-1,False
4,PAYMENT,2,4,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,Puerto Rico,...,1360,73,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class,2,-2,False
5,TRANSFER,6,4,294.980011,Shipping canceled,0,73,Sporting Goods,Tonawanda,EE. UU.,...,1360,73,Smart watch,327.75,0,2018-01-19 11:03:00,Standard Class,6,2,True
6,DEBIT,2,1,288.420013,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,...,1360,73,Smart watch,327.75,0,2018-01-15 10:42:00,First Class,2,1,True
7,TRANSFER,2,1,285.140015,Late delivery,1,73,Sporting Goods,Miami,EE. UU.,...,1360,73,Smart watch,327.75,0,2018-01-15 10:21:00,First Class,2,1,True
8,CASH,3,2,278.589996,Late delivery,1,73,Sporting Goods,Caguas,Puerto Rico,...,1360,73,Smart watch,327.75,0,2018-01-16 10:00:00,Second Class,3,1,True
9,CASH,2,1,275.309998,Late delivery,1,73,Sporting Goods,San Ramon,EE. UU.,...,1360,73,Smart watch,327.75,0,2018-01-15 09:39:00,First Class,2,1,True
